# Visualisation de la base DuckDB
Ce notebook permet de charger les données de `data/radar.duckdb` et `data/offres.parquet`, d’explorer les principales métriques et de visualiser les distributions ainsi que les relations entre variables.

In [ ]:
# Section 1: Connecter à la base de données
# Nous chargeons la base DuckDB et le fichier Parquet pour pouvoir explorer les données.

import duckdb
import pandas as pd
from pathlib import Path

DB_PATH = Path('data/radar.duckdb')
PARQUET_PATH = Path('data/offres.parquet')

print('DuckDB exists:', DB_PATH.exists())
print('Parquet exists:', PARQUET_PATH.exists())

conn = duckdb.connect(str(DB_PATH))
print('Connexion DuckDB établie:', conn)

# Vérifier les tables disponibles
print(conn.execute("SHOW TABLES").fetchall())

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Section 2: Charger les données dans un DataFrame
# On charge la table principale `offres` depuis DuckDB et on vérifie le chargement.

offres_df = conn.execute('SELECT * FROM offres').df()
print('Nombre de lignes:', len(offres_df))
print('Colonnes:', offres_df.columns.tolist())

offres_df.head()

In [ ]:
# Section 3: Explorer les données
# Aperçu général des colonnes, des types et des valeurs manquantes.

offres_df.info()

display(offres_df.describe(include='all').T)

missing = offres_df.isna().sum().sort_values(ascending=False)
print('Valeurs manquantes par colonne :')
print(missing[missing > 0])

In [ ]:
# Section 4: Visualiser les distributions
# Histogrammes et distribution des offres par date et par secteur.

# Convertir au besoin en datetime
for col in ['date_publication', 'date_creation', 'date_modification']:
    if col in offres_df.columns:
        offres_df[col] = pd.to_datetime(offres_df[col], errors='coerce')

fig = px.histogram(offres_df, x='date_publication', nbins=30, title='Distribution des dates de publication')
fig.show()

fig = px.bar(
    offres_df['secteur'].value_counts().reset_index(name='count'),
    x='count',
    y='index',
    orientation='h',
    title='Nombre d\'offres par secteur',
    labels={'index': 'Secteur', 'count': 'Nombre d\'offres'}
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()


In [ ]:
# Section 5: Visualiser les relations
# Corrélations et relations entre variables.

# Relation entre date de publication et secteur (nombre d'offres)
br_secteur = offres_df.groupby('secteur')['id'].count().reset_index(name='count')
fig = px.bar(br_secteur.sort_values('count', ascending=False).head(25),
             x='count', y='secteur', orientation='h',
             title='Top 25 des secteurs les plus actifs')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

# Heatmap de corrélation pour les colonnes numériques
numeric_cols = offres_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
if numeric_cols:
    corr = offres_df[numeric_cols].corr()
    fig = px.imshow(corr, text_auto=True, title='Heatmap de corrélation')
    fig.show()
else:
    print('Aucune colonne numérique disponible pour la corrélation.')

In [ ]:
# Section 6: Exporter ou sauvegarder les résultats
# Enregistrer un sous-ensemble de données et des statistiques.

OUTPUT_DIR = Path('notebooks/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUTPUT_DIR / 'offres_summary.csv'
offres_df.sample(min(1000, len(offres_df)), random_state=42).to_csv(summary_path, index=False)
print('Extrait sauvegardé:', summary_path)

# Sauvegarder un regroupement secteur -> nombre d'offres
secteur_counts = offres_df['secteur'].value_counts().reset_index()
secteur_counts.columns = ['secteur', 'count']
secteur_counts.to_csv(OUTPUT_DIR / 'secteur_counts.csv', index=False)
print('Regroupement secteur sauvegardé:', OUTPUT_DIR / 'secteur_counts.csv')

# Exemple d'export image via plotly
fig = px.bar(secteur_counts.sort_values('count', ascending=False).head(20),
             x='count', y='secteur', orientation='h',
             title='Top 20 secteurs')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.write_image(str(OUTPUT_DIR / 'top20_secteurs.png'))
print('Visualisation sauvegardée:', OUTPUT_DIR / 'top20_secteurs.png')